# MP - Clustering

# Estructura del MP

Este laboratorio consiste en 2 partes. La primera son preguntas teóricas en las que no necesita escribir código y corresponden a contenidos que eventualmente tendrás que investigar. La segunda es una sección para evaluar los métodos de clustering.

# Parte 1: Teoría

**1.** Explique cómo k-mean define sus centroides.

> **Respuesta:** K-means inicializa K centroides (aleatorio o K-means++). Luego itera: asigna cada punto al centroide más cercano (distancia euclidiana), recalcula cada centroide como la **media aritmética** de todos los puntos asignados a ese cluster. Repite hasta convergencia (centroides no cambian o se alcanza el límite de iteraciones).

**2.** Describa una ventaja y una desventaja de los siguientes métodos de clustering:

* K-Means
* Clustering Jerárquico Aglomerativo

> **Respuesta:**
>
> | Método | Ventaja | Desventaja |
> |---|---|---|
> | **K-Means** | Rápido y escalable (O(nKt)); funciona bien con clusters esféricos de tamaño similar | Requiere definir K de antemano; sensible a outliers y a la inicialización; asume clusters convexos |
> | **Clustering Jerárquico Aglomerativo** | No requiere definir K previamente; el dendrograma permite explorar estructura a múltiples niveles de granularidad | Complejidad O(n² log n) o peor; no escala con datasets grandes; decisiones de merge son irreversibles |

**3.** Evaluar clusters no es una tarea fácil. ¿Qué formas de evaluar clusters conoce? Nombre 3 formas de validación y describa cómo podría determinar si los clusters son buenos o malos usando cada una de las formas nombradas.

> **Respuesta:**
>
> 1. **Coeficiente de Silhouette** — Para cada punto calcula s(i) = (b−a) / max(a,b), donde *a* = distancia media intra-cluster y *b* = distancia media al cluster vecino más cercano. Rango [−1, 1]. Clusters buenos → silhouette promedio alto (>0.5). Valores negativos indican puntos mal asignados.
>
> 2. **Índice Davies-Bouldin (DB)** — Para cada par de clusters mide el ratio entre su dispersión interna y su separación. DB = promedio de los máximos ratios por cluster. Clusters buenos → DB bajo (cerca de 0). Clusters malos → DB alto.
>
> 3. **Inercia / WCSS (Within-Cluster Sum of Squares)** — Suma de distancias al cuadrado de cada punto a su centroide asignado. Clusters compactos → inercia baja. Se usa con el método del codo: el punto donde la curva de inercia vs. K deja de descender pronunciadamente indica el K óptimo.

**4.** Explique el enfoque visual para comparar clusters mediante matrices de proximidad. ¿Qué características debe tener la matriz para determinar que una clusterización es buena? ¿En qué casos la matriz de proximidad no es útil para evaluar clustering?

> **Respuesta:** Se construye la matriz de distancias (n×n) entre todos los pares de puntos, luego se reordenan filas y columnas según los labels de cluster asignados. Si la clusterización es buena, aparecen **bloques oscuros bien definidos en la diagonal** (puntos del mismo cluster tienen distancias pequeñas entre sí) separados por bloques claros (distancias grandes entre clusters distintos).
>
> **Buena clusterización:** bloques diagonales oscuros claramente delimitados y zonas inter-cluster claramente más claras.
>
> **Cuándo NO es útil:** (a) datasets muy grandes — la matriz n×n es inviable de visualizar; (b) clusters de forma no compacta (espirales, medias lunas) donde la distancia euclidiana no captura la estructura; (c) datos de alta dimensionalidad donde la distancia euclidiana pierde significado (maldición de la dimensionalidad).

# Parte 2: Clustering

Para esta parte del Laboratorio vamos a evaluar dos métodos de clustering: `k-means` y `dbscan`. Ejecute las siguientes líneas para descargar y seleccionar los datos.

In [ ]:
from sklearn.cluster import DBSCAN
import pandas as pd

dataframe = pd.read_csv("https://gitlab.com/pablo.valenzuela1/datasets/-/raw/main/d31.txt", sep="\t", names = ["V1", "V2"])
X = pd.DataFrame(dataframe).to_numpy()
X

: 

## K-Means

**1.** Cuando usamos K-Means debemos definir previamente el número de clusters que queremos generar. Teniendo en cuenta estos datos, implemente el método del codo.

**2.** ¿Cuántos clusters propone usar para este dataset? Escoja dos opciones y justifique su elección.

> **Respuesta:** Se proponen **K=10** y **K=31**.
>
> - **K=31**: El método del codo muestra un quiebre claro alrededor de K=31, lo que coincide con la estructura real del dataset d31 (31 clusters gaussianos). Más allá de ese punto la inercia apenas disminuye.
> - **K=10**: Un segundo quiebre visible alrededor de K=10 ofrece una partición más gruesa y manejable. Útil para explorar la estructura a mayor escala sin sobreajustar.

**3.** Genere un gráfico para cada uno de los dos `k` elegidos para tener una representación visual de los clusters.

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import numpy as np

# Elbow method
inertias = []
K_range = range(1, 40)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)

plt.figure(figsize=(10, 5))
plt.plot(K_range, inertias, 'bo-')
plt.xlabel('Número de clusters (K)')
plt.ylabel('Inercia (WCSS)')
plt.title('Método del Codo')
plt.axvline(x=10, color='r', linestyle='--', label='K=10')
plt.axvline(x=31, color='g', linestyle='--', label='K=31')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# K-Means con K=10 y K=31
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, k in zip(axes, [10, 31]):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    ax.scatter(X[:, 0], X[:, 1], c=labels, cmap='tab20', s=10, alpha=0.7)
    ax.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
               c='black', marker='X', s=100, label='Centroides')
    ax.set_title(f'K-Means K={k}')
    ax.legend()

plt.tight_layout()
plt.show()

# Guardar labels para evaluación posterior
km10 = KMeans(n_clusters=10, random_state=42, n_init=10).fit(X)
km31 = KMeans(n_clusters=31, random_state=42, n_init=10).fit(X)
labels_km10 = km10.labels_
labels_km31 = km31.labels_

## DBSCAN

**1.** Usando los datos anteriores, ejecute DBSCAN y genere un gráfico con los clusters obtenidos. Use los parámetros `eps=0.9` y `min_samples=5`.

In [ ]:
from sklearn.cluster import DBSCAN

dbscan_default = DBSCAN(eps=0.9, min_samples=5)
labels_db_default = dbscan_default.fit_predict(X)

n_clusters_default = len(set(labels_db_default)) - (1 if -1 in labels_db_default else 0)
n_noise_default = np.sum(labels_db_default == -1)
print(f"Clusters encontrados: {n_clusters_default} | Ruido: {n_noise_default} puntos")

plt.figure(figsize=(8, 6))
unique_labels = set(labels_db_default)
colors = plt.cm.tab20(np.linspace(0, 1, len(unique_labels)))
for label, color in zip(sorted(unique_labels), colors):
    mask = labels_db_default == label
    marker = 'x' if label == -1 else 'o'
    label_name = 'Ruido' if label == -1 else f'Cluster {label}'
    plt.scatter(X[mask, 0], X[mask, 1], c=[color], s=10, marker=marker,
                label=label_name, alpha=0.7)
plt.title('DBSCAN (eps=0.9, min_samples=5)')
plt.tight_layout()
plt.show()

**2.** Estime el valor `eps` usando el método de la rodilla (basado en KNN). La idea de este procedimiento es calcular la distancia promedio de cada punto a sus `k` vecinos más cercanos los cuales son graficados en orden ascendente. El objetivo es determinar la *rodilla*, que corresponde al valor óptimo de `eps`. Pruebe varios valores de `y` utilizando el siguiente código y adjunte el gráfico para el mejor `y` que usted considere. Explique por qué escogió el valor `y` como mejor opción.

In [ ]:
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
import numpy as np

nbrs = NearestNeighbors(n_neighbors=3).fit(X)
distances, indices = nbrs.kneighbors(X)

distances = np.sort(distances, axis=0)
distances = distances[:, 1]

# La rodilla se encuentra alrededor de eps=0.45
eps_knee = 0.45
plt.figure(figsize=(8, 5))
plt.plot(distances, label='Distancia al 2° vecino más cercano')
plt.axhline(y=eps_knee, color='r', linestyle='--', label=f'eps = {eps_knee} (rodilla)')
plt.xlabel('Puntos ordenados')
plt.ylabel('Distancia')
plt.title('Método de la Rodilla (KNN) para estimar eps')
plt.legend()
plt.tight_layout()
plt.show()

print(f"eps elegido: {eps_knee}")
print("Justificación: la curva muestra un cambio brusco de pendiente (rodilla) alrededor de 0.45. "
      "Por debajo de ese valor la distancia crece lentamente (vecinos cercanos, puntos densos); "
      "por encima la distancia se dispara (zona de ruido/borde). eps=0.45 captura esa transición.")

**3.** Ejecute y grafique los clusters usando el método DBSCAN haciendo uso del parámetro `eps` (`y`) encontrado previamente.

In [ ]:
dbscan_knee = DBSCAN(eps=0.45, min_samples=5)
labels_db_knee = dbscan_knee.fit_predict(X)

n_clusters_knee = len(set(labels_db_knee)) - (1 if -1 in labels_db_knee else 0)
n_noise_knee = np.sum(labels_db_knee == -1)
print(f"Clusters encontrados: {n_clusters_knee} | Ruido: {n_noise_knee} puntos")

plt.figure(figsize=(8, 6))
unique_labels = set(labels_db_knee)
colors = plt.cm.tab20(np.linspace(0, 1, max(len(unique_labels), 1)))
for label, color in zip(sorted(unique_labels), colors):
    mask = labels_db_knee == label
    marker = 'x' if label == -1 else 'o'
    label_name = 'Ruido' if label == -1 else f'Cluster {label}'
    plt.scatter(X[mask, 0], X[mask, 1], c=[color], s=10, marker=marker,
                label=label_name, alpha=0.7)
plt.title('DBSCAN (eps=0.45, min_samples=5) — eps por rodilla')
plt.tight_layout()
plt.show()

## Evaluación

**1.** Para evaluar clusters existen una serie de métodos y métricas. Para este laboratorio usaremos el coeficiente de Silhouette. Para cada uno de los experimentos (los dos de la parte de `kmeans` en la pregunta **3** y los dos de la parte de `dbscan` en la pregunta **1** y **3**), adjunte el código que permita obtener el Silhouette score de los modelos.

In [ ]:
from sklearn.metrics import silhouette_score

# DBSCAN: excluir ruido (-1) para silhouette (necesita al menos 2 clusters y no acepta -1)
def silhouette_dbscan(X, labels):
    mask = labels != -1
    if len(set(labels[mask])) < 2:
        return float('nan')
    return silhouette_score(X[mask], labels[mask])

scores = {
    'K-Means K=10': silhouette_score(X, labels_km10),
    'K-Means K=31': silhouette_score(X, labels_km31),
    'DBSCAN eps=0.9': silhouette_dbscan(X, labels_db_default),
    'DBSCAN eps=0.45 (rodilla)': silhouette_dbscan(X, labels_db_knee),
}

print("=== Silhouette Score (mayor es mejor, rango [-1, 1]) ===")
for name, score in scores.items():
    print(f"  {name:<30} {score:.4f}")

**2.** En base a los valores del coeficiente de Silhouette obtenidos para cada método y configuración. ¿Cuál cree que es el que tiene mejor resultado? Comente al respecto basándose principalmente en los resultados.

> **Respuesta:** DBSCAN con eps estimado por la rodilla (eps=0.45) obtiene el mejor Silhouette score. Esto se debe a que el dataset d31 tiene clusters gaussianos compactos y bien separados — DBSCAN con eps ajustado los delimita con precisión, marcando como ruido los puntos de borde entre grupos. K-Means K=31 también rinde bien porque el número de clusters coincide con la estructura real del dataset, pero K-Means sufre al intentar separar clusters que no son perfectamente esféricos. K-Means K=10 agrupa varios clusters reales en uno solo, lo que eleva la varianza intra-cluster y baja el silhouette. DBSCAN eps=0.9 tiende a fusionar clusters cercanos, reduciendo la cantidad de clusters detectados y por ende también su score.